In [2]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings("ignore")

# Load sample reviews
reviews = pd.read_csv("../data/processed/reviews_clean.csv", low_memory=False)
reviews = reviews[reviews["comments"].notna()]
reviews = reviews[reviews["comments"].str.len() > 30]
reviews = reviews[~reviews["comments"].str.contains("nan", na=False)]

# Sample 5000 for RAG corpus
corpus = reviews.sample(5000, random_state=42).reset_index(drop=True)
print(f"RAG corpus: {len(corpus)} reviews")
print(corpus["comments"].iloc[0][:200])

RAG corpus: 5000 reviews
Had a great stay at the accommodation, just felt abit worried about the no short stay policy, loved the coffee next door and cafes in the area. Only downside was neighbours who were smoking cannabis… 


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text

# Build TF-IDF index
print("Building TF-IDF index...")
corpus["clean_comments"] = corpus["comments"].apply(clean_text)

vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    stop_words="english"
)
tfidf_matrix = vectorizer.fit_transform(corpus["clean_comments"])
print(f"TF-IDF matrix: {tfidf_matrix.shape}")

def retrieve(query, top_k=5):
    query_clean = clean_text(query)
    query_vec = vectorizer.transform([query_clean])
    scores = cosine_similarity(query_vec, tfidf_matrix).flatten()
    top_idx = scores.argsort()[-top_k:][::-1]
    results = []
    for idx in top_idx:
        results.append({
            "score": round(scores[idx], 4),
            "listing_id": corpus.iloc[idx]["listing_id"],
            "review": corpus.iloc[idx]["comments"][:200]
        })
    return results

# Test retrieval
print("\n--- Query: 'cleanliness problems dirty room' ---")
for r in retrieve("cleanliness problems dirty room"):
    print(f"Score: {r['score']} | Listing: {r['listing_id']}")
    print(f"  {r['review']}\n")

Building TF-IDF index...
TF-IDF matrix: (5000, 5000)

--- Query: 'cleanliness problems dirty room' ---
Score: 0.4056 | Listing: 15777680
  Smell was bad and room was dirty.

Score: 0.3607 | Listing: 30577801
  location and cleanliness of room were really good.

Score: 0.3133 | Listing: 680113462757980981
  Overall,  the location was convenient and the room was as described in the description. Hosts were responsive and friendly.<br/>The only major concern was cleanliness-  the kettle was unwashed and had

Score: 0.3114 | Listing: 7341294
  great location. other than bathroom being dirty it was as advertised

Score: 0.2735 | Listing: 32683658
  So pleasently surprised with this place. The cleanliness, the little touches that Darren adds to his space is awesome. So much more room than expected and really felt like home. The cleanliness was am



In [4]:
from collections import Counter

def answer_question(question, top_k=10):
    """Simple RAG: retrieve relevant reviews, synthesize answer"""
    results = retrieve(question, top_k=top_k)
    
    # Aggregate insights from retrieved reviews
    all_text = " ".join([r["review"] for r in results])
    
    # Extract key words
    words = clean_text(all_text).split()
    stopwords = {"the","a","an","and","or","was","is","are","were","be",
                 "have","has","had","it","its","this","that","very","so",
                 "as","by","from","we","our","i","my","they","their","you",
                 "your","not","no","do","did","just","up","out","if","about"}
    keywords = [w for w in words if w not in stopwords and len(w) > 3]
    top_keywords = Counter(keywords).most_common(8)
    
    print(f"\n{'='*60}")
    print(f"Q: {question}")
    print(f"{'='*60}")
    print(f"Top {top_k} relevant reviews retrieved.")
    print(f"Key themes: {', '.join([k for k,v in top_keywords])}")
    print(f"\nMost relevant review:")
    print(f"  \"{results[0]['review']}\"")
    print(f"\nSupporting evidence:")
    for r in results[1:4]:
        print(f"  [{r['score']}] {r['review'][:150]}")

# Test Q&A
answer_question("What do guests say about location and transport?")
answer_question("What are the main complaints about cleanliness?")
answer_question("What do guests love most about their stay?")


Q: What do guests say about location and transport?
Top 10 relevant reviews retrieved.
Key themes: location, guests, good, stay, great, host, recommend, other

Most relevant review:
  "Very good location, cleen room,  the rooftop is beautiful. Definitely i would say i enjoyed my time.<br/>Thank youu!"

Supporting evidence:
  [0.2211] A neat hideout with impeccable location, convenient transport within walkable reach, would recommend a stay.
  [0.218] Great location. Check in was easy and smooth, no hassle. I didn't stay long enough to be able to comment about socialising with other guests, but the 
  [0.1983] The room and host's service were good for the price. I can recommend you to stay here. However, I'll explain their location for future guests as some 

Q: What are the main complaints about cleanliness?
Top 10 relevant reviews retrieved.
Key themes: cleanliness, stay, complaints, place, great, location, room, would

Most relevant review:
  "Great stay as always. Zero complaints."

In [5]:
print("RAG SYSTEM EVALUATION")
print("="*50)
print("""
Architecture:
- Retrieval: TF-IDF vectorization with cosine similarity
- Corpus: 5,000 sampled Bangkok Airbnb reviews
- Query processing: bigram TF-IDF (1,2 ngrams)

Strengths:
1. No API key required — fully local and reproducible
2. Fast retrieval (<1 second for 5,000 docs)
3. Relevant results for domain-specific queries
4. Transparent scoring via cosine similarity

Limitations:
1. TF-IDF lacks semantic understanding — 
   'complaints about cleanliness' retrieves 'zero complaints'
   because it matches on keywords not meaning
2. 5,000 review sample — full 513K corpus would improve coverage
3. No LLM synthesis layer — answers are extracted, not generated
4. No multilingual support — German/Thai reviews not handled

Production Improvements:
1. Replace TF-IDF with sentence-transformers embeddings
   (e.g. all-MiniLM-L6-v2) for semantic search
2. Add LLM synthesis layer (Claude/GPT-4) to generate
   natural language answers from retrieved context
3. Scale to full corpus using FAISS vector index
4. Add language detection and translation preprocessing
""")

RAG SYSTEM EVALUATION

Architecture:
- Retrieval: TF-IDF vectorization with cosine similarity
- Corpus: 5,000 sampled Bangkok Airbnb reviews
- Query processing: bigram TF-IDF (1,2 ngrams)

Strengths:
1. No API key required — fully local and reproducible
2. Fast retrieval (<1 second for 5,000 docs)
3. Relevant results for domain-specific queries
4. Transparent scoring via cosine similarity

Limitations:
1. TF-IDF lacks semantic understanding — 
   'complaints about cleanliness' retrieves 'zero complaints'
   because it matches on keywords not meaning
2. 5,000 review sample — full 513K corpus would improve coverage
3. No LLM synthesis layer — answers are extracted, not generated
4. No multilingual support — German/Thai reviews not handled

Production Improvements:
1. Replace TF-IDF with sentence-transformers embeddings
   (e.g. all-MiniLM-L6-v2) for semantic search
2. Add LLM synthesis layer (Claude/GPT-4) to generate
   natural language answers from retrieved context
3. Scale to full co

In [6]:
# --- 7.4 Generative AI & Agentic Experimentation ---

print("""
GENERATIVE AI & AGENTIC EXPERIMENTATION
=========================================

7.4.1 Dynamic Pricing Advisor Design
--------------------------------------
An AI-powered dynamic pricing advisor would work as follows:

INPUT SIGNALS:
- Current listing price vs neighbourhood median
- Occupancy rate trend (last 30/60/90 days)
- Seasonal demand index (from calendar data)
- Competitor pricing in same neighbourhood
- Days until booking window opens
- Special events calendar (festivals, conferences)

PRICING RECOMMENDATIONS:
- If occupancy > 70% and price < neighbourhood median:
  → Increase price by 10-15% (underpriced for demand)
- If occupancy < 20% and availability > 60 days:
  → Decrease price by 10% or offer weekly discount
- If peak season approaching (Jul-Sep):
  → Apply 15-25% seasonal premium
- If weekend and no weekend premium set:
  → Apply 5-10% Friday-Saturday surcharge

IMPLEMENTATION:
- Rule-based engine as baseline
- ML model (Gradient Boosting) for price elasticity
- LLM layer for natural language explanation of recommendations
- Daily retraining on new calendar/booking data

7.4.2 AI-Generated Listing Descriptions
-----------------------------------------
Experiment: Compare AI-generated vs human-written descriptions

Approach:
- Extract key listing attributes (room type, amenities, location)
- Prompt Claude/GPT-4 to generate marketing description
- Compare word count, sentiment score, and booking conversion

Sample prompt used:
"Generate a compelling Airbnb listing description for a 
1-bedroom entire apartment in Vadhana, Bangkok priced at 
฿1,500/night with pool, gym, BTS access, and city views. 
Target: international tourists. Tone: warm, professional."

Finding: AI descriptions tend to be more structured and 
keyword-rich but less authentic than host-written descriptions. 
Human descriptions containing personal touches and local 
knowledge may drive higher trust and conversion rates.

7.4.3 Responsible AI Framework
--------------------------------
For deploying ML models in an Airbnb-like marketplace:

1. TRANSPARENCY
   - All price recommendations explained in plain language
   - Hosts can override any AI suggestion
   - Model confidence scores disclosed

2. FAIRNESS
   - Regular bias audits across neighbourhoods and host types
   - Separate models for different market segments
   - No protected characteristics in feature set

3. PRIVACY
   - Guest data anonymized before model training
   - No individual guest profiling
   - GDPR/PDPA compliant data handling

4. ACCOUNTABILITY
   - Human review required for >20% price change recommendations
   - Audit trail for all model decisions
   - Regular model performance reporting to stakeholders

5. SAFETY
   - Price floor/ceiling guardrails (±50% from market median)
   - Anomaly detection for unusual pricing patterns
   - Rollback mechanism if model degrades

7.4.4 MLOps Workflow Design
-----------------------------
Continuous model retraining pipeline:

Data Ingestion (weekly scrape)
        ↓
Data Validation (Great Expectations)
        ↓
Feature Engineering (automated)
        ↓
Model Training (scheduled Airflow DAG)
        ↓
Model Evaluation (MAE/RMSE vs baseline)
        ↓
A/B Testing (shadow deployment)
        ↓
Production Deployment (if metrics pass)
        ↓
Monitoring (drift detection, alerting)
        ↓
Retraining trigger (if drift detected)
""")


GENERATIVE AI & AGENTIC EXPERIMENTATION

7.4.1 Dynamic Pricing Advisor Design
--------------------------------------
An AI-powered dynamic pricing advisor would work as follows:

INPUT SIGNALS:
- Current listing price vs neighbourhood median
- Occupancy rate trend (last 30/60/90 days)
- Seasonal demand index (from calendar data)
- Competitor pricing in same neighbourhood
- Days until booking window opens
- Special events calendar (festivals, conferences)

PRICING RECOMMENDATIONS:
- If occupancy > 70% and price < neighbourhood median:
  → Increase price by 10-15% (underpriced for demand)
- If occupancy < 20% and availability > 60 days:
  → Decrease price by 10% or offer weekly discount
- If peak season approaching (Jul-Sep):
  → Apply 15-25% seasonal premium
- If weekend and no weekend premium set:
  → Apply 5-10% Friday-Saturday surcharge

IMPLEMENTATION:
- Rule-based engine as baseline
- ML model (Gradient Boosting) for price elasticity
- LLM layer for natural language explanation of

In [9]:
# --- LLM-Powered Listing Improvement Recommendations ---
# Using Claude API via Anthropic

import anthropic
import json

# Sample a few listings with their reviews
df_listings_sample = pd.read_csv("../data/processed/listings_enriched.csv", 
                                  low_memory=False)
reviews_full = pd.read_csv("../data/processed/reviews_clean.csv", 
                            low_memory=False)

# Pick 3 listings with multiple reviews
listing_ids = df_listings_sample[
    df_listings_sample["number_of_reviews"].between(10, 50)
]["id"].sample(3, random_state=42).tolist()

client = anthropic.Anthropic()

for listing_id in listing_ids:
    listing = df_listings_sample[df_listings_sample["id"] == listing_id].iloc[0]
    listing_reviews = reviews_full[
        reviews_full["listing_id"] == listing_id
    ]["comments"].dropna().head(5).tolist()
    
    if not listing_reviews:
        continue
    
    review_text = "\n".join([f"- {r[:200]}" for r in listing_reviews])
    
    prompt = f"""You are a short-term rental consultant analyzing Bangkok Airbnb listings.

Listing details:
- Room type: {listing['room_type']}
- Price: ฿{listing['price']}/night
- Neighbourhood: {listing['neighbourhood_cleansed']}
- Rating: {listing['review_scores_rating']}
- Number of reviews: {listing['number_of_reviews']}

Recent guest reviews:
{review_text}

Provide 3 specific, actionable improvement recommendations for this listing host.
Be concise — one sentence per recommendation.
Format as: 1. [recommendation] 2. [recommendation] 3. [recommendation]"""

    message = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=300,
        messages=[{"role": "user", "content": prompt}]
    )
    
    print(f"\nListing ID: {listing_id}")
    print(f"Type: {listing['room_type']} | Price: ฿{listing['price']} | Rating: {listing['review_scores_rating']}")
    print(f"Recommendations:")
    print(message.content[0].text)
    print("-"*50)